In [ ]:
#
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.
#


# 🕵️ BigQuery ML for Tax Anomaly Detection
This notebook demonstrates a complete end-to-end workflow using **BigQuery ML** to train, evaluate, and predict anomalies in tax data. 

---

## 1. Setup and Environment
Install necessary Python packages to interact with BigQuery.

In [ ]:
### Install Dependencies
!pip install google-auth google-cloud-bigquery pandas db-dtypes tabulate

## 2. Initialize BigQuery Client
Import libraries, define project constants, and establish a connection to BigQuery, respecting the specified universe domain.

In [ ]:
import google.auth
from google.cloud import bigquery
from google.api_core.client_options import ClientOptions
import warnings

# Suppress non-critical BigQuery warnings
warnings.filterwarnings("ignore", message=".*BigQuery Storage module not found.*")


import os
# 🛠️ Configuration
PROJECT_ID = 'PLACEHOLDER_PROJECT_ID'
UNIVERSE_DOMAIN = 'PLACEHOLDER_UNIVERSE_DOMAIN'
DATASET_ID = 'tax_office_dataset'
MODEL_ID = 'anomaly_model'
TABLE_ID = 'tax_data_table'
PREDICTION_VIEW_ID= 'predictions'

# Get credentials and configure the client options
credentials, default_project = google.auth.default()
api_endpoint = f"https://bigquery.{UNIVERSE_DOMAIN}"
client_options = ClientOptions(api_endpoint=api_endpoint)

# Create the BigQuery client instance
client = bigquery.Client(
  project=PROJECT_ID,
  credentials=credentials,
  client_options=client_options
)

print(f"✅ BigQuery Client initialized for project: {PROJECT_ID}")

## 3. Train Anomaly Detection Model
We use the `CREATE MODEL` statement to train a **Logistic Regression** model on the tax data table using a variety of financial, behavioral, and crypto-related features to predict the `is_anomaly` label.

In [ ]:
create_model_query = f"""
CREATE OR REPLACE MODEL `{PROJECT_ID}.{DATASET_ID}.{MODEL_ID}`
OPTIONS(
  MODEL_TYPE='LOGISTIC_REG',
  INPUT_LABEL_COLS=['is_anomaly']
) AS
SELECT
  -- 🏷️ Basic features
  taxpayer_type,
  industry_code,
  address_state,

  -- 📈 Financial ratios
  deduction_ratio,
  tax_ratio,
  crypto_income_ratio,

  -- 🏃 Behavioral features
  is_late_filing,
  has_amendments,
  is_late_payment,
  days_payment_delay,
  days_filing_delay,

  -- 🕰️ Historical patterns
  late_filing_rate,
  late_payment_rate,
  amendment_rate,
  income_volatility,
  avg_deduction_ratio,

  -- ₿ Cryptocurrency features
  has_crypto_account,
  has_declared_crypto_previously,
  is_crypto_non_declarant,
  crypto_transaction_count,
  crypto_risk_score,

  -- 🎯 Target
  is_anomaly
FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}`
WHERE is_anomaly IS NOT NULL
  AND data_source = 'TRAINING' -- Use only training data for model creation
"""

# Run the query
print("⏳ Creating and training the model... This may take a few moments.")
query_job = client.query(create_model_query)
query_job.result()  # Wait for the query to finish

print(f"✅ Model `{MODEL_ID}` created and trained successfully in BigQuery.")

## 4. Evaluate Model Performance
Use `ML.EVALUATE` to retrieve the model's metrics and print the results.

In [ ]:
evaluate_query = f"""
SELECT * FROM ML.EVALUATE(MODEL `{PROJECT_ID}.{DATASET_ID}.{MODEL_ID}`)
"""

# Run the query and load results into a DataFrame
evaluate_df = client.query(evaluate_query).to_dataframe()

print("📊 Model Evaluation Results:")
print(evaluate_df.to_markdown(index=False))
# Optional: Use evaluate_df.to_html() if you prefer HTML output in the notebook

## 5. Generate Predictions
Use `ML.PREDICT` on the non-training data and store the results in a new BigQuery View. The results are ordered by anomaly probability.

In [ ]:
predict_query = f"""
CREATE OR REPLACE VIEW `{PROJECT_ID}.{DATASET_ID}.{PREDICTION_VIEW_ID}` AS
SELECT
  taxpayer_id,
  declaration_id,
  predicted_is_anomaly,
  -- Extract the probability for the predicted class (TRUE or FALSE)
  CASE
    WHEN predicted_is_anomaly = TRUE THEN predicted_is_anomaly_probs[OFFSET(0)].prob
    ELSE predicted_is_anomaly_probs[OFFSET(1)].prob
  END AS anomaly_probability
FROM ML.PREDICT(MODEL `{PROJECT_ID}.{DATASET_ID}.{MODEL_ID}`,
  (SELECT * FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_ID}` WHERE data_source != 'TRAINING')
)
ORDER BY predicted_is_anomaly DESC, anomaly_probability DESC
"""

# Run the query to create the view
print("⏳ Generating predictions and creating the view...")
client.query(predict_query).result() # Wait for view creation

# Fetch a sample of the prediction results
sample_query = f"SELECT * FROM `{PROJECT_ID}.{DATASET_ID}.{PREDICTION_VIEW_ID}` LIMIT 10"
predict_df = client.query(sample_query).to_dataframe()

print(f"✅ Prediction View `{PREDICTION_VIEW_ID}` created. Showing top 10 anomalies:")
print(predict_df.to_markdown(index=False))

## 🎉 Conclusion
You have successfully trained an anomaly detection model and generated predictions using BigQuery ML, all managed from this Jupyter Notebook.